In [ ]:
# pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_core-1.3.0-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached xxhash-3.6.0-cp313-cp313-win_amd64.whl.metadata (13 kB)
  Using cached orjson-3.11.8-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached zstandard-0.25.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached sqlalchemy-2.0.49-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached aiohttp-3.13.5-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_core.documents import Document
from langchain_community.document_loaders.text import TextLoader


e:\Pytorch_ML\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sample_doc = Document(
    page_content="This is a sample document.",
    metadata={"source": "sample.pdf"},
)   
print(sample_doc.page_content)
print(sample_doc.metadata)

This is a sample document.
{'source': 'sample.pdf'}


In [4]:
text_loader = TextLoader("data/data.text", encoding="utf-8")
documents = text_loader.load()

In [5]:
print(documents)

[Document(metadata={'source': 'data/data.text'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mor

In [6]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research2.pdf")
# documents = pdf_loader.load()
# documents

Ingestion pipline

In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader


In [8]:
def load_pdf_documents(pdf_dir):
    folder_path = os.path.join(os.getcwd(), pdf_dir)
    documents = []
    all_files = os.listdir(folder_path)
    pdf_files = [f for f in all_files if f.endswith('.pdf')]

    for pdf_file in pdf_files:
        file_path = os.path.join(folder_path, pdf_file)
        loader = PyPDFLoader(file_path)
        docs = loader.load()
        documents.extend(docs)
    return documents

Chunks

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size = 500, chunk_overlap = 50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap
    )

    split_docs = text_splitter.split_documents(documents)
    return split_docs


    


In [10]:
chunked_docs = split_doc(documents)
len(chunked_docs)

5

Embeddings

In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7276.63it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
class EmbeddingModel:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model_name = model_name
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        print(self.model.get_sentence_embedding_dimension())

    def generate_embedding(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        return embeddings


In [13]:
embedding_model = EmbeddingModel()

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10976.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


384


C:\Users\aksha\AppData\Local\Temp\ipykernel_1060\3524327979.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(self.model.get_sentence_embedding_dimension())


In [14]:
import chromadb
import uuid
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11685.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        existing_collections = [col.name for col in self.client.list_collections()]

        if self.collection_name in existing_collections:
            self.collection = self.client.get_collection(name=self.collection_name)
        else:
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={"description": "vector store for cosine similarity"}
            )

        print(f"Vector store initialized at: {self.persist_directory}")
        print(f"Collection count: {self.collection.count()} documents")


    def add_documents(self, documents, embeddings):
        texts = [doc.page_content for doc in documents]

        if len(texts) != len(embeddings):
            raise ValueError("Mismatch between texts and embeddings")

        ids = []
        metadatas = []
        embeddings_list = []

        for doc, embedding in zip(documents, embeddings):
            ids.append(f"doc_{uuid.uuid4()}")

            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["source"] = metadata.get("source", "unknown")
            metadatas.append(metadata)

            # safe conversion
            embeddings_list.append(
                embedding.tolist() if hasattr(embedding, "tolist") else embedding
            )

        self.collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings_list,
            metadatas=metadatas
        )

        print(f"Added {len(texts)} documents")
        print(f"Total in DB: {self.collection.count()}")

In [16]:
vectore_store = VectorStoreManager()

Vector store initialized at: data/vector_store
Collection count: 10 documents


In [17]:
texts = [doc.page_content for doc in chunked_docs]
embeddings = embedding_model.encode(texts)
# embeddings = embedding_model.generate_embedding(texts)
vectore_store.add_documents(chunked_docs, embeddings)


Added 5 documents
Total in DB: 15


In [18]:
class RAGPipeline:
    def __init__(self, vector_store_manager):
        self.vector_store_manager = vector_store_manager    
        self.vector_store = vector_store_manager.collection

    def ask(self, query):
        query_embedding = embedding_model.encode([query])[0]
        results = self.vector_store.query(
            query_embeddings=[query_embedding],
            n_results=5,
            include=["documents", "metadatas"]
        )
        return results 
    
rag_pipeline = RAGPipeline(vectore_store)
query = "What is the main topic of the documents?"
results = rag_pipeline.ask(query)
results

{'ids': [['doc_374cd11a-6c41-4f60-8ed3-058953be7028',
   'doc_b0cd1bf6-1645-4cbc-b37d-3c1116822f86',
   'doc_e7f04b63-6a2c-4852-8dde-baa8b4953766',
   'doc_856906c3-dbcb-4e18-aef7-f485d7ff418d',
   'doc_688a52f2-cc76-437c-bf7a-ed8f938c36ef']],
 'embeddings': None,
 'documents': [['Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports',
   'Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes 

Integrate with LLMs

In [31]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)

In [42]:
def generate_output(query, vector_store, llm, top_k=3):
    # Chroma Collection has query(), not retrieve()
    query_embedding = embedding_model.encode([query])[0]
    results = vector_store.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas"]
    )

    docs = results.get("documents", [[]])[0] if results else []
    context = "\n".join(docs) if docs else ""

    if not context:
        print("No relevant context found for the query.")
        return "No relevant context found."

    prompt = f"""Use the given context to answer the question.

Context:
{context}

Question:
{query}"""

    response = llm.invoke(prompt)
    return response.content

In [45]:
ask_query = "What is RAG?"
response = generate_output(ask_query, rag_pipeline.vector_store, llm)
print(response)

<think>
Okay, the user is asking, "What is RAG?" Let me start by recalling what I know about RAG. RAG stands for Retrieval-Augmented Generation. It's a technique used in natural language processing, especially in large language models. The idea is to combine information retrieval with text generation. So when a model needs to answer a question, it first retrieves relevant documents from a database and then uses that information to generate a more accurate and up-to-date response.

Wait, but the context provided here is about Python's rapid prototyping capabilities. The user included context that mentions Python's interpreted nature, simplicity, and ecosystem. However, the question is about RAG, which isn't directly related to Python's features. The context doesn't mention RAG at all. 

Hmm, maybe the user expects me to connect Python's features to RAG? Like how Python is used in implementing RAG systems? But the question is just asking for the definition of RAG. The context given doesn